# Milestone 2 Lab, notebook 6 of 6: The pipeline, and the same question again

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the business's records | Recall the question and the answer key, and explore the records profit is calculated from |
| 2. One calculation, two engines | Write a calculation by hand in pandas and in SQL, with test cases that judge both |
| 3. A model drafts a transformation | Ask a model to write the calculation, and let the tests judge it |
| 4. Which model? Accuracy, cost, and speed | Give the same tasks to several models and compare them |
| 5. An agent that tests and fixes its own code | Give a model the tests as a tool, and compare it with a single draft |
| **6. The pipeline, and the same question again** (you are here) | Run the calculations on each engine and ask the analyst the leader's question |

Model requests in this notebook: **about 8**. A full run of this lab costs well under a dollar.

## What this notebook is for

Everything comes together here.

1. The reviewed transformations run as a **pipeline**, on either engine.
2. The calculated figures are checked against milestone 1's answer key.
3. The analyst from milestone 1 answers the leader's question again, this time on calculated figures and with real records as evidence.

If this works, you have replaced a mock with the real thing, and the person asking the question never noticed.

In [1]:
import pandas as pd
from lab_helpers import (
    LAST_MONTH,
    MAIN_QUESTION,
    check_key_against_calculated,
    load_result,
    show_answer,
    show_doc,
)

from retail_profit_stack import tools
from retail_profit_stack.agent import run_analyst
from retail_profit_stack.evaluation import load_cases, run_evaluation, summarise
from retail_profit_stack.store import get_store
from retail_profit_stack.transform.contracts import tables_differ
from retail_profit_stack.transform.pipeline import run_pipeline

## Step 1: Run the pipeline

The **pipeline** loads the business's records, runs the six transformations in order (each one after the ones it depends on), and saves the results. It is one function call.

In [2]:
show_doc("HLD.md", "3. Components and data flow")

> **From `docs/HLD.md`: 3. Components and data flow**

*(Diagram: open `docs/HLD.md` to see it.)*

In [3]:
run = run_pipeline("sql")

print(f"Ran on the {run.engine} engine in {run.seconds} seconds")
pd.DataFrame([{"table": name, "rows": count} for name, count in run.row_counts.items()])

Ran on the sql engine in 0.08 seconds


,table,rows
0,sales,24
1,refund_totals,24
2,delivery_costs,24
3,fee_totals,24
4,product_cost,24
5,monthly_product_profit,24


Here is the profit table it produced. It has the same columns as the table that was prepared by hand in milestone 1. This time every number has a calculation behind it.

In [4]:
profit = run.tables["monthly_product_profit"].to_pandas()

profit[profit["month"] == LAST_MONTH][["product_id", "units_sold", "net_sales", "product_cost", "gross_profit", "contribution_profit"]].sort_values("contribution_profit", ascending=False)

,product_id,units_sold,net_sales,product_cost,gross_profit,contribution_profit
12,P001,510,18252.00,9180.0,9072.00,5188.20
19,P008,340,10170.00,3740.0,6430.00,4250.00
20,P009,150,16240.00,10500.0,5740.00,3841.00
17,P006,88,12397.50,7216.0,5181.50,3297.30
15,P004,165,8516.75,3960.0,4556.75,3129.11
23,P012,175,10901.25,6300.0,4601.25,2934.06
13,P002,240,7946.75,3600.0,4346.75,2541.35
14,P003,120,9639.00,5760.0,3879.00,2499.40
21,P010,480,7743.60,2880.0,4863.60,2464.32
16,P005,36,10080.00,6840.0,3240.00,1818.96


The camp stove (P007) is at the bottom with no cost and no profit, as the rule requires. Its sales are still there.

## Step 2: Change the engine

One word changes the engine. Nothing else does. Do both engines produce the same six tables?

In [5]:
other = run_pipeline("pandas", warehouse_dir=None)

pd.DataFrame([{"table": name, "difference between engines": tables_differ(other.tables[name], run.tables[name]) or "none"} for name in run.tables])

,table,difference between engines
0,sales,none
1,refund_totals,none
2,delivery_costs,none
3,fee_totals,none
4,product_cost,none
5,monthly_product_profit,none


This is what the head of data asked for in the BRD: the rules do not change when the technology does. When data grows enough to need PySpark, a third engine slots in the same way, and these same checks will say whether it is right.

## Step 3: Does it match the answer key?

The answer key was worked out by hand in milestone 1 from the prepared table. The figures below are read from the calculated table, through the same tools the analyst uses.

In [6]:
get_store.cache_clear()  # make the tools load the tables the pipeline just wrote
answer_key = load_result("answer_key")

check_key_against_calculated(answer_key)

,What,Answer key (milestone 1),Calculated (milestone 2),Same?
0,#1 product: Trail water bottle,"$5,188.20","$5,188.20",yes
1,#2 product: Headlamp,"$4,250.00","$4,250.00",yes
2,#3 product: Rain jacket,"$3,841.00","$3,841.00",yes
3,Top category: Hydration,"$13,358.06","$13,358.06",yes
4,"Hydration, change since the month before",+$376.06,+$376.06,yes
5,"Apparel, change since the month before (the bi...","-$5,914.87","-$5,914.87",yes
6,"Camp, change since the month before (cannot be...",unavailable,unavailable,yes
7,Camp stove (its cost is missing),unavailable,unavailable,yes


Every line should say **yes**. The calculations reproduce, from about 2,500 order lines, the figures that milestone 1 took on trust.

## Step 4: Evidence is real now

In milestone 1 the analyst's evidence was a few hand-written sample rows. Now `get_evidence` returns real records from the business's tables, with the most notable ones first.

In [7]:
evidence = tools.get_evidence(LAST_MONTH, "P001", limit=8)

print(f"{evidence.total_records} records lie behind this product's figures. {evidence.note}")
pd.DataFrame([record.model_dump() for record in evidence.records])[["record_type", "source_record_id", "amount", "description"]]

763 records lie behind this product's figures. These are real records from the business's source tables, most notable first. They are some of the records behind the figures, not all of them.


,record_type,source_record_id,amount,description
0,order_item,OI-01246,36.00,1 x Trail water bottle on order ORD-21207 (202...
1,refund,REF-0032,36.00,Refund issued 2026-10-05 against line OI-01065...
2,product_cost,P001/2026-10,18.00,Unit cost agreed with the vendor for 2026-10
3,shipment,SHP-01246,6.50,Shipped 2026-10-02 for line OI-01246: pick and...
4,payment_fee,FEE-01246,1.08,Payment fee captured 2026-10-01 for line OI-01246
5,order_item,OI-01246,36.00,1 x Trail water bottle on order ORD-21207 (202...
6,refund,REF-0033,36.00,Refund issued 2026-10-05 against line OI-01252...
7,shipment,SHP-01246,6.50,Shipped 2026-10-02 for line OI-01246: pick and...


Look at the first and sixth rows: the same order line, twice. That is one of the duplicated lines behind quality flag QF-002, surfaced as evidence.

## Step 5: The same question again

Now the moment of truth. The analyst is exactly as you left it in milestone 1: the same six tools, the same instructions, the same verifier. Only the data behind the tools has changed.

In [8]:
print(MAIN_QUESTION)

analyst_run = run_analyst(MAIN_QUESTION)

print(f"\nAnswered in {analyst_run.seconds} seconds with {len(analyst_run.tool_log)} tool calls")

Which products and categories generated the most contribution profit last month, what changed, and how confident should I be in those numbers?

Answered in 14.3 seconds with 7 tool calls


> **If that cell reports an error**, run it again. It needs your OpenRouter key in the `.env` file, as in milestone 1.

In [9]:
show_answer(analyst_run.answer)

### In 2026-10, Trail water bottle led contribution profit at $5,188.20, followed by Headlamp at $4,250.00. Apparel fell 37.8% versus 2026-09, while Hydration rose 2.9%. Camp is incomplete because Camp stove has no product cost, so confidence is low.
Top products by contribution profit in 2026-10: Trail water bottle $5,188.20, Headlamp $4,250.00, Rain jacket $3,841.00, Sleeping bag $3,297.30, Water filter $3,129.11. Camp stove is unavailable (null) because its product cost is missing (QF-001), so it is not ranked. Categories: Hydration $13,358.06, Apparel $9,746.38, and Camp $9,366.26. The Camp total excludes Camp stove and is incomplete. Month-on-month, Apparel fell $5,914.87 (-37.8%), driven by Rain jacket (-$2,234.30, -36.8%) and Sun hat (-$1,632.50, -76.3%). Hydration rose $376.06 (+2.9%), led by Hydration pack (+$628.25) and Trail water bottle (+$85.00, +1.7%, possibly overstated per QF-002). Sleeping bag (+$860.55) and Headlamp (+$538.00) were the largest product gains. Camp stove's September value was $2,886.50, but October cannot be compared. The Camp category change is also unavailable.

| Figure | Value | Month |
|---|---|---|
| Trail water bottle, contribution profit | $5,188.20 | 2026-10 |
| Headlamp, contribution profit | $4,250.00 | 2026-10 |
| Rain jacket, contribution profit | $3,841.00 | 2026-10 |
| Sleeping bag, contribution profit | $3,297.30 | 2026-10 |
| Water filter, contribution profit | $3,129.11 | 2026-10 |
| Camp stove, contribution profit | unavailable | 2026-10 |
| Hydration, contribution profit | $13,358.06 | 2026-10 |
| Apparel, contribution profit | $9,746.38 | 2026-10 |
| Camp, contribution profit (incomplete, excludes Camp stove) | $9,366.26 | 2026-10 |
| Rain jacket, contribution profit change vs 2026-09 | -$2,234.30 | 2026-10 |
| Rain jacket, contribution profit percent change vs 2026-09 | -36.8% | 2026-10 |
| Sun hat, contribution profit change vs 2026-09 | -$1,632.50 | 2026-10 |
| Sun hat, contribution profit percent change vs 2026-09 | -76.3% | 2026-10 |
| Apparel, contribution profit change vs 2026-09 | -$5,914.87 | 2026-10 |
| Apparel, contribution profit percent change vs 2026-09 | -37.8% | 2026-10 |
| Hydration, contribution profit change vs 2026-09 | +$376.06 | 2026-10 |
| Hydration, contribution profit percent change vs 2026-09 | +2.9% | 2026-10 |
| Hydration pack, contribution profit change vs 2026-09 | +$628.25 | 2026-10 |
| Sleeping bag, contribution profit change vs 2026-09 | +$860.55 | 2026-10 |
| Headlamp, contribution profit change vs 2026-09 | +$538.00 | 2026-10 |
| Trail water bottle, contribution profit change vs 2026-09 | +$85.00 | 2026-10 |

- **Caveat QF-001:** Camp stove has no product cost in 2026-10. Its gross profit and contribution profit are unavailable, and the Camp category total leaves it out.
- **Caveat QF-002:** Three Trail water bottle orders appear twice in 2026-10. Its sales and profit may be overstated, so its rank and the +$85.00 change may be unreliable.

**Confidence: low.** QF-001 (high): Camp stove has no product cost in 2026-10, so its profit is missing and the Camp total is incomplete. QF-002 (medium): Trail water bottle's three duplicated orders may overstate its sales and profit, which affects the top-ranked product.

And the verifier's report, the same seven checks as in milestone 1.

In [10]:
print("Verified:", analyst_run.verification.passed)
pd.DataFrame([check.model_dump() for check in analyst_run.verification.checks])

Verified: True


,name,passed,detail
0,figures_match_tools,True,21 figures match tool results
1,no_unsupported_numbers,True,Every number in the text comes from a tool result
2,sources_cited,True,"Cites ['metric_definitions', 'monthly_product_..."
3,definitions_stated,True,Every metric used has its definition and versi...
4,confidence_matches_rule,True,Answer says low; the rule gives low
5,flags_mentioned,True,All 2 returned flags are in the caveats
6,decline_is_clean,True,Not a declined answer


The leader gets the same answer as before. What changed is what stands behind it.

| | Milestone 1 | Milestone 2 |
|---|---|---|
| Profit figures | Typed in by hand | Calculated from the business's records |
| Can they be reproduced? | No | Yes, by running the pipeline |
| Evidence | Hand-written samples | Real records |
| Can the technology change? | Not applicable | Yes: two engines today, a third when needed |

## Step 6: The evaluation set

Milestone 1's seven evaluation questions must still pass. Running them uses the analyst for real, about 15 model requests for the three "tiny" cases. It is off by default.

In [ ]:
RUN_EVALUATION = False

if RUN_EVALUATION:
    scores = run_evaluation(run_analyst, load_cases(tiny=True))
    for score in scores:
        failed = [check.name for check in score.checks if not check.passed]
        print(f"{score.case_id}: {'passed' if score.passed else 'failed ' + str(failed)}")
        if score.error:
            print("   The model did not return an answer:", score.error)
    display(pd.Series(summarise(scores), name="passed").to_frame())
else:
    print("Evaluation not run. Set RUN_EVALUATION = True to run it.")

## Step 7: Run the tests

Open a terminal in this milestone's folder (`src/m2_metric_calculation_layer`) and run:

```
uv run pytest
```

Among other things, the tests check that every transformation passes every hand-worked case on both engines, that the engines agree, that the calculated table equals milestone 1's, and that unsafe code from a model is refused.

To run the pipeline yourself from the terminal, on either engine:

```
uv run python -m retail_profit_stack.transform.pipeline sql
uv run python -m retail_profit_stack.transform.pipeline pandas
```

> ### Your assignment
>
> 1. **Break a rule and watch it get caught.** Open `retail_profit_stack/transform/implementations/sql/product_cost.sql` and change `LEFT JOIN` to `JOIN`. Run `uv run pytest`. Which tests fail, and what do their messages say? Change it back.
> 2. **Add a test case.** In `evals/m2_transform_cases.json`, add a case for `sales` with two lines of different products on one order. Work out the expected output by hand first.
> 3. **Try another model in notebook 4.** Add a model to the `MODELS` list and re-run. Where does it land on the accuracy against cost chart?
> 4. **Stretch.** Read `retail_profit_stack/transform/engines/spark_engine.py`. What exactly would you need to write to make the PySpark engine real?

> ### What you built
>
> 1. You met the business's records, problems and all.
> 2. You wrote a calculation by hand in two languages, and built the test cases that judge it.
> 3. You had a model draft a transformation, and let the tests decide.
> 4. You compared several models on accuracy, cost, and speed, with evidence.
> 5. You gave a model the tests as a tool and watched it fix its own work.
> 6. You ran the reviewed transformations as a pipeline, on two engines, and the leader got the same answer.
>
> The lesson to carry forward: **a model can write the code, but you decide what correct means.** Write the rule down as a contract, work out test cases by hand, let those judge every draft, and review what passes.
>
> Two things are still mocked. The data problems are still a hand-written list, and the records are still generated. In Milestone 3 real quality checks take over from that list, and the duplicated lines finally get dealt with.

**One last thing.** Open `milestone_2_recap.html`, in this milestone's folder, in your browser.